# Production Benchmark Analysis: NN Field Map Variants

**Goal:** Apples-to-apples comparison of trilinear interpolation vs NN field map variants,
run through the full LHCb `TrackRungeKuttaExtrapolator` pipeline (including Jacobian,
adaptive step-size control, covariance transport).

**Variants benchmarked:**
1. Trilinear interpolation (baseline)
2. NN scalar SiLU [32]
3. NN scalar ReLU [32]
4. NN AVX2 ReLU [32]

**Data source:** `benchmark_nn_field_map.root` produced by `benchmark_nn_field_map.py`
using `TrackExtrapolatorTesterSOA` with `EnableSubTimers=True`.

In [ ]:
import uproot
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

plt.rcParams.update({
    'figure.figsize': (12, 7),
    'font.size': 13,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

## 1. Load ROOT ntuple data

In [ ]:
ROOT_FILE = Path('../tests/options/benchmark_nn_field_map.root')
# Adjust path as needed based on where gaudirun.py writes output
if not ROOT_FILE.exists():
    # Try build directory
    for candidate in [
        Path('benchmark_nn_field_map.root'),
        Path('../build/benchmark_nn_field_map.root'),
    ]:
        if candidate.exists():
            ROOT_FILE = candidate
            break

print(f'Looking for: {ROOT_FILE.resolve()}')
assert ROOT_FILE.exists(), f'ROOT file not found: {ROOT_FILE}  — run the benchmark first'

f = uproot.open(ROOT_FILE)
print('Keys:', f.keys())

In [ ]:
VARIANTS = ['Trilinear', 'NN_SiLU', 'NN_ReLU', 'NN_AVX2_ReLU']
COLORS = {'Trilinear': '#1f77b4', 'NN_SiLU': '#ff7f0e', 'NN_ReLU': '#2ca02c', 'NN_AVX2_ReLU': '#d62728'}

dfs = {}
for v in VARIANTS:
    # Locate the ntuple — key format may vary
    for key in f.keys():
        if v in key:
            tree = f[key]
            dfs[v] = tree.arrays(library='pd')
            print(f'{v}: {len(dfs[v])} tracks, columns: {list(dfs[v].columns)[:10]}...')
            break
    else:
        print(f'WARNING: {v} not found in ROOT file')

print(f'\nLoaded {len(dfs)} variants')

## 2. Per-Track Timing Overview

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 2a. Histogram of per-track wall-clock time
ax = axes[0]
for v in VARIANTS:
    if v not in dfs:
        continue
    df = dfs[v]
    t = df['time']  # nanoseconds
    ax.hist(t, bins=50, alpha=0.5, label=f'{v} (med={np.median(t):.0f} ns)', color=COLORS[v])
ax.set_xlabel('Per-track propagation time (ns)')
ax.set_ylabel('Tracks')
ax.set_title('Wall-clock time distribution')
ax.legend(fontsize=10)

# 2b. Box plot comparison
ax = axes[1]
data_bp = [dfs[v]['time'].values for v in VARIANTS if v in dfs]
labels_bp = [v for v in VARIANTS if v in dfs]
bp = ax.boxplot(data_bp, labels=labels_bp, patch_artist=True)
for patch, v in zip(bp['boxes'], labels_bp):
    patch.set_facecolor(COLORS[v])
    patch.set_alpha(0.6)
ax.set_ylabel('Time (ns)')
ax.set_title('Per-track timing comparison')

plt.tight_layout()
plt.savefig('timing_overview.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Summary table
rows = []
baseline_med = np.median(dfs['Trilinear']['time']) if 'Trilinear' in dfs else 1
for v in VARIANTS:
    if v not in dfs:
        continue
    t = dfs[v]['time']
    rows.append({
        'Variant': v,
        'Median (ns)': f'{np.median(t):.0f}',
        'Mean (ns)': f'{np.mean(t):.0f}',
        'Std (ns)': f'{np.std(t):.0f}',
        'Min (ns)': f'{np.min(t):.0f}',
        'P95 (ns)': f'{np.percentile(t, 95):.0f}',
        'Speedup': f'{baseline_med / np.median(t):.2f}x',
    })
pd.DataFrame(rows)

## 3. Sub-Operation Timing Breakdown (RDTSC cycles)

In [ ]:
SUB_OPS = ['field_cycles', 'deriv_cycles', 'butcher_cycles', 'jacobian_cycles', 'stepsize_cycles']
SUB_LABELS = ['Field eval', 'Derivatives', 'Butcher accum', 'Jacobian', 'Step-size ctrl']

fig, axes = plt.subplots(1, len([v for v in VARIANTS if v in dfs]), figsize=(4*len(VARIANTS), 5), sharey=True)
if not hasattr(axes, '__len__'):
    axes = [axes]

for ax, v in zip(axes, [v for v in VARIANTS if v in dfs]):
    df = dfs[v]
    if 'field_cycles' not in df.columns:
        ax.set_title(f'{v}\n(no sub-timers)')
        continue
    means = [df[op].mean() for op in SUB_OPS]
    other = df['total_cycles'].mean() - sum(means)
    means.append(max(0, other))
    labels = SUB_LABELS + ['Other']
    colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00', '#999999']
    ax.barh(labels, means, color=colors)
    ax.set_xlabel('Mean cycles per propagation')
    ax.set_title(v)
    # Annotate percentages
    total = sum(means)
    for i, (m, l) in enumerate(zip(means, labels)):
        if total > 0:
            ax.text(m + total*0.02, i, f'{m:.0f} ({100*m/total:.0f}%)', va='center', fontsize=9)

plt.suptitle('Sub-operation breakdown per propagation', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('sub_operation_breakdown.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Per-step field evaluation cost
rows = []
for v in VARIANTS:
    if v not in dfs or 'field_cycles' not in dfs[v].columns:
        continue
    df = dfs[v]
    nsteps = df['nsteps']
    field = df['field_cycles']
    total = df['total_cycles']
    # CK stages per step: 6 field evaluations per accepted step
    field_per_call = field / (nsteps * 6)
    rows.append({
        'Variant': v,
        'Mean steps': f'{nsteps.mean():.1f}',
        'Mean rejected': f"{df['nrejected'].mean():.1f}",
        'Field cycles/call': f'{field_per_call.mean():.0f}',
        'Field % of total': f'{100 * field.mean() / total.mean():.1f}%',
        'Total cycles': f'{total.mean():.0f}',
    })
pd.DataFrame(rows)

## 4. Accuracy: Position & Slope Residuals vs Trilinear

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
resid_cols = [('dx', 'x residual (mm)'), ('dy', 'y residual (mm)'),
              ('dtx', 'tx residual'), ('dty', 'ty residual')]

for ax, (col, label) in zip(axes.flat, resid_cols):
    for v in VARIANTS:
        if v not in dfs or col not in dfs[v].columns:
            continue
        vals = dfs[v][col]
        ax.hist(vals, bins=100, alpha=0.5, label=f'{v} (RMS={np.std(vals):.4g})',
                color=COLORS[v], density=True)
    ax.set_xlabel(label)
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)

plt.suptitle('Residuals vs trilinear reference', fontsize=14)
plt.tight_layout()
plt.savefig('accuracy_residuals.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Accuracy summary table
rows = []
for v in VARIANTS:
    if v not in dfs:
        continue
    df = dfs[v]
    rows.append({
        'Variant': v,
        'RMS dx (mm)': f"{np.std(df['dx']):.4g}" if 'dx' in df.columns else 'N/A',
        'RMS dy (mm)': f"{np.std(df['dy']):.4g}" if 'dy' in df.columns else 'N/A',
        'RMS dtx': f"{np.std(df['dtx']):.6g}" if 'dtx' in df.columns else 'N/A',
        'RMS dty': f"{np.std(df['dty']):.6g}" if 'dty' in df.columns else 'N/A',
        'Max |dx| (mm)': f"{np.max(np.abs(df['dx'])):.4g}" if 'dx' in df.columns else 'N/A',
        'Max |dy| (mm)': f"{np.max(np.abs(df['dy'])):.4g}" if 'dy' in df.columns else 'N/A',
    })
pd.DataFrame(rows)

## 5. Stacked Timing: Where Does Time Go?

In [ ]:
# Stacked bar chart: mean cycle breakdown across variants
fig, ax = plt.subplots(figsize=(10, 6))

variants_with_timers = [v for v in VARIANTS if v in dfs and 'field_cycles' in dfs[v].columns]
x = np.arange(len(variants_with_timers))
width = 0.6

bottoms = np.zeros(len(variants_with_timers))
bar_colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00', '#999999']

for i, (op, label) in enumerate(zip(SUB_OPS + ['other'], SUB_LABELS + ['Other'])):
    vals = []
    for v in variants_with_timers:
        df = dfs[v]
        if op == 'other':
            total = df['total_cycles'].mean()
            sub_sum = sum(df[o].mean() for o in SUB_OPS)
            vals.append(max(0, total - sub_sum))
        else:
            vals.append(df[op].mean())
    vals = np.array(vals)
    ax.bar(x, vals, width, bottom=bottoms, label=label, color=bar_colors[i])
    bottoms += vals

ax.set_xticks(x)
ax.set_xticklabels(variants_with_timers, fontsize=11)
ax.set_ylabel('Mean cycles per propagation')
ax.set_title('Stacked sub-operation timing')
ax.legend(loc='upper right')

# Annotate total
for i, v in enumerate(variants_with_timers):
    ax.text(i, bottoms[i] + bottoms.max()*0.02, f'{bottoms[i]:.0f}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('stacked_timing.pdf', bbox_inches='tight')
plt.show()

## 6. Field Evaluation Cost vs Total: Amdahl's Law

In [ ]:
# Show how field eval fraction determines the maximum possible speedup
fig, ax = plt.subplots(figsize=(8, 5))

for v in variants_with_timers:
    df = dfs[v]
    field_frac = df['field_cycles'] / df['total_cycles']
    ax.hist(field_frac * 100, bins=30, alpha=0.5, label=v, color=COLORS[v])

ax.set_xlabel('Field evaluation as % of total propagation')
ax.set_ylabel('Tracks')
ax.set_title('Amdahl\'s Law: field eval fraction determines speedup ceiling')
ax.legend()
ax.axvline(50, color='grey', ls='--', alpha=0.5, label='50% → max 2× speedup')

plt.tight_layout()
plt.savefig('amdahl_field_fraction.pdf', bbox_inches='tight')
plt.show()

## 7. Timing vs Track Properties

In [ ]:
# Timing vs q/p (momentum dependence)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for v in VARIANTS:
    if v not in dfs:
        continue
    df = dfs[v]
    if 'qop' not in df.columns:
        continue
    ax = axes[0]
    ax.scatter(np.abs(df['qop'])*1e3, df['time'], alpha=0.15, s=3, label=v, color=COLORS[v])
    ax.set_xlabel('|q/p| (1/GeV)')
    ax.set_ylabel('Time (ns)')
    ax.set_title('Timing vs momentum')

axes[0].legend(markerscale=5)

# Nsteps vs q/p
for v in VARIANTS:
    if v not in dfs or 'nsteps' not in dfs[v].columns:
        continue
    df = dfs[v]
    ax = axes[1]
    ax.scatter(np.abs(df['qop'])*1e3, df['nsteps'], alpha=0.15, s=3, label=v, color=COLORS[v])
    ax.set_xlabel('|q/p| (1/GeV)')
    ax.set_ylabel('Number of RK steps')
    ax.set_title('Steps vs momentum')

axes[1].legend(markerscale=5)
plt.tight_layout()
plt.savefig('timing_vs_momentum.pdf', bbox_inches='tight')
plt.show()

## 8. Summary Table (LaTeX)

In [ ]:
# Generate LaTeX-ready summary table
print(r'\begin{table}[htbp]')
print(r'\centering')
print(r'\caption{Production benchmark: NN field map variants vs trilinear interpolation.}')
print(r'\label{tab:production_benchmark}')
print(r'\begin{tabular}{lrrrrr}')
print(r'\toprule')
print(r'Variant & Median (ns) & Speedup & RMS $\Delta x$ (mm) & Field \% & Steps \\')
print(r'\midrule')

baseline_med = np.median(dfs['Trilinear']['time']) if 'Trilinear' in dfs else 1
for v in VARIANTS:
    if v not in dfs:
        continue
    df = dfs[v]
    med = np.median(df['time'])
    speedup = baseline_med / med
    rms_dx = np.std(df['dx']) if 'dx' in df.columns else float('nan')
    field_pct = 100 * df['field_cycles'].mean() / df['total_cycles'].mean() if 'field_cycles' in df.columns else float('nan')
    nsteps = df['nsteps'].mean() if 'nsteps' in df.columns else float('nan')
    name = v.replace('_', r'\_')
    print(f'{name} & {med:.0f} & {speedup:.2f}$\\times$ & {rms_dx:.4g} & {field_pct:.1f}\\% & {nsteps:.1f} \\\\')

print(r'\bottomrule')
print(r'\end{tabular}')
print(r'\end{table}')

## 9. Key Findings

Fill in after running the benchmark:

1. **Field evaluation fraction of total:** The sub-operation breakdown reveals what percentage
   of total propagation time is spent in field evaluation. This sets the Amdahl's Law ceiling
   for any field-eval speedup.

2. **NN vs trilinear wall-clock:** Direct median timing comparison shows the actual speedup
   (or slowdown) when replacing trilinear interpolation with NN inference.

3. **AVX2 benefit:** The scalar ReLU vs AVX2 ReLU comparison isolates the SIMD speedup
   within the full production pipeline.

4. **Accuracy trade-off:** RMS position/slope residuals quantify the physics cost of
   the NN approximation.